<a href="https://colab.research.google.com/github/basilhakim/NLP-classworks/blob/main/Q2solarpowergeneration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 **ASSIGNMENT 1 (QUESTION 2)**

MUHAMMAD BASIL BIN ABDUL HAKIM

2215315


### Independent Variable Selection

For predicting solar generated power, the most suitable independent variables are those directly influencing solar panel efficiency and sunlight availability. Based on this, the following variables were selected:

*   **`shortwave_radiation_backwards_sfc`**: This represents the amount of solar radiation reaching the surface. More radiation directly leads to higher power generation.
*   **`zenith`** and **`angle_of_incidence`**: These describe the sun's position relative to the solar panel. The angle at which sunlight hits the panel significantly impacts how much energy is absorbed.
*   **`total_cloud_cover_sfc`**: Cloud cover obstructs sunlight, reducing the amount of radiation reaching the panels and thus decreasing power generation.
*   **`temperature_2_m_above_gnd`**: While solar panels need sunlight, very high temperatures can reduce their efficiency. This variable accounts for ambient temperature effects.

These variables are fundamental physical factors that directly dictate the potential for solar power generation, making them highly relevant for the prediction model.

In [ ]:
# Import Libraries and Load Solar Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the dataset (Make sure you renamed the file to spg.csv in Colab!)
df_solar = pd.read_excel('spg.xlsx')

# Display missing values to confirm
print("Missing values before cleaning:\n", df_solar.isnull().sum()[df_solar.isnull().sum() > 0])

Missing values before cleaning:
 temperature_2_m_above_gnd            1
shortwave_radiation_backwards_sfc    1
wind_direction_10_m_above_gnd        1
generated_power_kw                   1
dtype: int64


In [ ]:
# Data Preprocessing for Solar Data
# Handling Missing Values
# There is 1 missing values in temperature, radiation, wind_direction, and generated_power.
# Because this is likely time-series weather data, we will fill missing values with the median of their respective columns.
for col in df_solar.columns:
    if df_solar[col].isnull().sum() > 0:
        df_solar[col] = df_solar[col].fillna(df_solar[col].median())

# Removing unnecessary/zero-variance variables (Optional but good practice)
# If any column has all 0s, it doesn't help the model.
# In this dataset, all variables have numerical variance, so we keep them.
# There are no categorical variables in this dataset, so we skip One-Hot Encoding!

# Define Independent (X) and Dependent (y) variables
X_solar = df_solar.drop('generated_power_kw', axis=1)
y_solar = df_solar['generated_power_kw']

# Splitting the Data
# 80% for training, 20% for testing
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_solar, y_solar, test_size=0.2, random_state=42)

# Feature Scaling
# Variables like pressure (1035) and wind speed (5) have vastly different scales.
scaler_s = StandardScaler()
X_train_scaled_s = scaler_s.fit_transform(X_train_s)
X_test_scaled_s = scaler_s.transform(X_test_s)

print("\nData preparation complete!")
print(f"Training data shape: {X_train_scaled_s.shape}")
print(f"Testing data shape: {X_test_scaled_s.shape}")


Data preparation complete!
Training data shape: (3370, 20)
Testing data shape: (843, 20)


In [ ]:
# Model Building and Evaluation for Solar Data
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# --- Part C: Multiple Linear Regression ---
mlr_solar = LinearRegression()
mlr_solar.fit(X_train_scaled_s, y_train_s)
y_pred_mlr_s = mlr_solar.predict(X_test_scaled_s)

print("--- Part C: Multiple Linear Regression Results ---")
print(f"R²: {r2_score(y_test_s, y_pred_mlr_s):.4f}")
print(f"MAE: {mean_absolute_error(y_test_s, y_pred_mlr_s):.2f} kW")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_s, y_pred_mlr_s)):.2f} kW\n")


# --- Part D: Polynomial Regression ---
# We use only the core drivers of solar power to prevent the curse of dimensionality
core_solar = ['shortwave_radiation_backwards_sfc', 'zenith', 'angle_of_incidence',
              'total_cloud_cover_sfc', 'temperature_2_m_above_gnd']
X_train_core_s = X_train_s[core_solar]
X_test_core_s = X_test_s[core_solar]

# Scale core features separately
scaler_core_s = StandardScaler()
X_train_c_scaled = scaler_core_s.fit_transform(X_train_core_s)
X_test_c_scaled = scaler_core_s.transform(X_test_core_s)

# Apply Polynomial Features (Degree = 3)
poly_s = PolynomialFeatures(degree=3)
X_train_poly_s = poly_s.fit_transform(X_train_c_scaled)
X_test_poly_s = poly_s.transform(X_test_c_scaled)

poly_model_s = LinearRegression()
poly_model_s.fit(X_train_poly_s, y_train_s)
y_pred_poly_s = poly_model_s.predict(X_test_poly_s)

print("--- Part D: Polynomial Regression (Degree=3) Results ---")
print(f"R²: {r2_score(y_test_s, y_pred_poly_s):.4f}")
print(f"MAE: {mean_absolute_error(y_test_s, y_pred_poly_s):.2f} kW")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_s, y_pred_poly_s)):.2f} kW\n")


# --- Part E: Support Vector Regression (SVR) ---
# We use the RBF kernel and increase C to 5000 to match the scale of the target variable.
svr_solar = SVR(kernel='rbf', C=5000)
svr_solar.fit(X_train_scaled_s, y_train_s)
y_pred_svr_s = svr_solar.predict(X_test_scaled_s)

print("--- Part E: Support Vector Regression (RBF) Results ---")
print(f"R²: {r2_score(y_test_s, y_pred_svr_s):.4f}")
print(f"MAE: {mean_absolute_error(y_test_s, y_pred_svr_s):.2f} kW")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_s, y_pred_svr_s)):.2f} kW")

--- Part C: Multiple Linear Regression Results ---
R²: 0.7180
MAE: 391.82 kW
RMSE: 507.58 kW

--- Part D: Polynomial Regression (Degree=3) Results ---
R²: 0.7282
MAE: 349.39 kW
RMSE: 498.31 kW

--- Part E: Support Vector Regression (RBF) Results ---
R²: 0.7866
MAE: 264.98 kW
RMSE: 441.55 kW


### Part F: Model Comparison

To determine the best-performing model, we compare the evaluation metrics (R², MAE, RMSE) for each developed model:

*   **Multiple Linear Regression (MLR):**
    *   R²: 0.7180
    *   MAE: 391.82 kW
    *   RMSE: 507.58 kW

*   **Polynomial Regression (Degree=3):**
    *   R²: 0.7282
    *   MAE: 349.39 kW
    *   RMSE: 498.31 kW

*   **Support Vector Regression (SVR - RBF kernel, C=5000):**
    *   R²: 0.7866
    *   MAE: 264.98 kW
    *   RMSE: 441.55 kW

#### Comparison and Justification:

Based on these metrics, the **Support Vector Regression (SVR) model** performs the best for this dataset.

1.  **Accuracy (R²):** SVR has the highest R² value (0.7866), indicating that it explains a larger proportion of the variance in the generated power compared to MLR (0.7180) and Polynomial Regression (0.7282). This suggests that SVR's predictions are closer to the actual values.

2.  **Error Metrics (MAE & RMSE):** SVR also achieves the lowest Mean Absolute Error (MAE: 264.98 kW) and Root Mean Squared Error (RMSE: 441.55 kW). Lower MAE implies smaller average absolute differences between predicted and actual values, while lower RMSE indicates smaller average squared differences, giving more weight to larger errors. Both metrics confirm SVR's superior predictive accuracy.

3.  **Generalization Ability:** Given its higher R² and lower error metrics on the test set, SVR demonstrates better generalization ability, meaning it is more likely to perform well on unseen data. The use of the RBF kernel allows SVR to capture non-linear relationships in the data effectively, which is often crucial for complex datasets like weather-dependent solar power generation.

4.  **Practical Considerations:** While SVR models can sometimes be computationally more intensive to train than linear models, their superior performance in this case justifies the potential increase in complexity, especially for critical applications like solar power forecasting where accuracy is paramount to optimize grid management and economic benefits. The chosen C parameter (5000) also indicates an effort to minimize error aggressively, balancing between fitting the training data well and avoiding excessive complexity through the kernel function.

In conclusion, SVR is the most robust and accurate model among the three for predicting solar generated power in this scenario.

In [ ]:
# Export Solar Results for Power BI
results_solar_df = pd.DataFrame({
    'Actual_Power_kW': y_test_s,
    'SVR_Prediction': y_pred_svr_s
})

# Add back some original weather data to filter by in Power BI
original_test_solar = df_solar.loc[y_test_s.index, ['shortwave_radiation_backwards_sfc', 'total_cloud_cover_sfc', 'temperature_2_m_above_gnd']]
powerbi_solar = pd.concat([original_test_solar, results_solar_df], axis=1)

# Calculate error
powerbi_solar['SVR_Error'] = abs(powerbi_solar['Actual_Power_kW'] - powerbi_solar['SVR_Prediction'])

# Export
powerbi_solar.to_csv("PowerBI_Solar_Results.csv", index=False)
print("Data successfully exported! Download 'PowerBI_Solar_Results.csv'")

Data successfully exported! Download 'PowerBI_Solar_Results.csv'
